# News Article Collection & Preprocessing
**NLP Final Project — Multi-Outlet Summarization Dataset**

Fetches news articles from three English-language outlets on non-overlapping topics:
- 🇬🇧 **BBC News** (UK)
- 🇦🇺 **ABC News Australia** (AU)
- 🇺🇸 **Associated Press** (US) — NewsAPI has no distinct "ABC News (US)" source, so AP stands in as the US outlet

Articles are saved to `articles/<topic>/<source>/article_NNN.json` and a flat CSV `articles/articles_dataset.csv` is produced for downstream NLP tasks.

In [1]:
# Install dependencies if needed
# %pip install requests pandas trafilatura newspaper3k lxml_html_clean nltk


In [2]:
import requests
import json
import os
import re
import time
import warnings
import pandas as pd
from datetime import datetime
from collections import defaultdict

warnings.filterwarnings('ignore')

# --- trafilatura: primary full-text extractor ---
try:
    import trafilatura
    TRAFILATURA_AVAILABLE = True
    print("trafilatura available  ✓")
except ImportError:
    TRAFILATURA_AVAILABLE = False
    print("trafilatura not found — run: pip install trafilatura")

# --- newspaper3k: fallback extractor ---
try:
    from newspaper import Article
    NEWSPAPER_AVAILABLE = True
    print("newspaper3k available  ✓")
except ImportError:
    NEWSPAPER_AVAILABLE = False
    print("newspaper3k not found  (optional fallback)")

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


trafilatura available  ✓
newspaper3k available  ✓


## Configuration

In [3]:
NEWS_API_KEY  = "b0d72f0b8c4f41d1b25c3d2cabeece44"
NEWS_API_BASE = "https://newsapi.org/v2"

SOURCES = {
    "bbc": {
        "source_id": "bbc-news",
        "domain":    "bbc.co.uk",
        "country":   "GB",
        "label":     "BBC News (UK)"
    },
    "abc_au": {
        "source_id": "abc-news-au",
        "domain":    "abc.net.au",
        "country":   "AU",
        "label":     "ABC News (Australia)"
    },
    "ap_us": {
        # NewsAPI has no distinct "ABC News (US)" source — Associated Press stands
        # in as the US outlet. Options: "associated-press", "cnn", "nbc-news", "the-washington-post"
        "source_id": "associated-press",
        "domain":    "apnews.com",
        "country":   "US",
        "label":     "Associated Press (US)"
    }
}

TOPICS = [
    "artificial intelligence",
    "public health",
    "elections",
    "global economy",
    "energy",
]

TARGET_PER_SOURCE = 10   # articles wanted per (topic × source) pair — 5 topics × 3 sources × 10 = 150 total
PAGE_SIZE         = 100  # articles per API request (NewsAPI max)
MAX_PAGES         = 3    # page cap per query — stops earlier if source exhausted

OUTPUT_DIR      = os.path.join(os.path.dirname(os.path.abspath("get_news.ipynb")), "articles")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "fetch_progress.json")

print(f"Target            : {TARGET_PER_SOURCE} articles per (topic × source)")
print(f"Topics            : {len(TOPICS)}  ({', '.join(TOPICS)})")
print(f"Expected total    : {len(TOPICS) * len(SOURCES) * TARGET_PER_SOURCE} articles")
print(f"Output directory  : {OUTPUT_DIR}")
print(f"Checkpoint file   : {CHECKPOINT_FILE}")

Target            : 10 articles per (topic × source)
Topics            : 5  (artificial intelligence, public health, elections, global economy, energy)
Expected total    : 150 articles
Output directory  : /Users/david/Documents/MLDS/Sem 4/NLP/FInal Project /articles
Checkpoint file   : /Users/david/Documents/MLDS/Sem 4/NLP/FInal Project /articles/fetch_progress.json


In [4]:
def validate_sources():
    """
    Hit /v2/sources to confirm every configured source_id exists on this API key.
    Run this before fetching — it costs only 1 request.
    """
    resp = requests.get(
        f"{NEWS_API_BASE}/sources",
        params={"apiKey": NEWS_API_KEY, "language": "en"},
        timeout=10
    )
    if resp.status_code != 200:
        print(f"Could not reach /v2/sources ({resp.status_code}). Check your API key.")
        return

    available = {s["id"]: s["name"] for s in resp.json().get("sources", [])}

    print(f"NewsAPI reports {len(available)} sources available on your plan.\n")
    all_ok = True
    for key, info in SOURCES.items():
        sid = info["source_id"]
        if sid in available:
            print(f"  ✓  {key:<10} '{sid}'  →  {available[sid]}")
        else:
            print(f"  ✗  {key:<10} '{sid}'  NOT FOUND — articles will return empty!")
            all_ok = False

    if not all_ok:
        print("\nReplace the missing source_id values in the SOURCES dict above.")
        print("Commonly available US sources: associated-press, cnn, nbc-news, the-washington-post, reuters")

validate_sources()

NewsAPI reports 78 sources available on your plan.

  ✓  bbc        'bbc-news'  →  BBC News
  ✓  abc_au     'abc-news-au'  →  ABC News (AU)
  ✓  ap_us      'associated-press'  →  Associated Press


## Directory Setup

In [5]:
def setup_directories():
    """Create articles/<topic>/<source>/ folder structure."""
    for topic in TOPICS:
        topic_slug = topic.lower().replace(" ", "_")
        for source_key in SOURCES:
            path = os.path.join(OUTPUT_DIR, topic_slug, source_key)
            os.makedirs(path, exist_ok=True)
    # Also a top-level folder for headlines
    os.makedirs(os.path.join(OUTPUT_DIR, "top_headlines"), exist_ok=True)
    print(f"Directory structure ready under: {OUTPUT_DIR}")

setup_directories()

Directory structure ready under: /Users/david/Documents/MLDS/Sem 4/NLP/FInal Project /articles


## NewsAPI Fetching Functions

In [6]:
class RateLimitError(Exception):
    pass


def _get(endpoint, params):
    """NewsAPI GET wrapper. Raises RateLimitError when the daily quota is exhausted."""
    params["apiKey"] = NEWS_API_KEY
    resp = requests.get(f"{NEWS_API_BASE}/{endpoint}", params=params, timeout=15)

    if resp.status_code == 429:
        raise RateLimitError("HTTP 429 — daily request quota exhausted")

    if resp.status_code == 200:
        data = resp.json()
        if data.get("status") == "ok":
            articles = data.get("articles", [])
            if not articles:
                print(f"    [0 results — totalResults={data.get('totalResults', '?')}, "
                      f"source={params.get('sources', '?')}, q={params.get('q', '—')}]")
            return articles, data.get("totalResults", 0)
        if data.get("code") == "rateLimited":
            raise RateLimitError(data.get("message", "Rate limit reached"))
        print(f"  API warning ({data.get('code')}): {data.get('message')}")
        return [], 0

    try:
        msg = resp.json().get("message", resp.text[:120])
    except Exception:
        msg = resp.text[:120]
    print(f"  HTTP {resp.status_code}: {msg}")
    return [], 0


def fetch_by_topic(source_id, query, target, page_size, max_pages, exclude_urls=None):
    """
    Paginate /everything to collect up to `target` unique articles.
    Stops as soon as the target is met or the source is exhausted.
    Raises RateLimitError if the API quota is hit mid-fetch.

    `exclude_urls` seeds the dedup set with URLs already saved to disk for this
    (topic, source) pair. NewsAPI's /everything with sortBy=relevancy returns the
    same top results deterministically for a repeated query — without this, resuming
    the fetch loop after a rate limit (or just re-running the notebook) re-fetches
    and re-saves articles that are already on disk under new article_NNN numbers.
    That bug is what produced the ~44% duplicate rate found and cleaned up in
    articles/ and summary_articles/ — see the dedup note in the markdown above.
    """
    collected = []
    seen_urls = set(exclude_urls or ())

    for page in range(1, max_pages + 1):
        batch, total = _get("everything", {
            "sources":  source_id,
            "q":        query,
            "pageSize": page_size,
            "page":     page,
            "language": "en",
            "sortBy":   "relevancy"
        })
        time.sleep(0.3)

        for art in batch:
            url = art.get("url", "")
            if url and url not in seen_urls:
                seen_urls.add(url)
                collected.append(art)
                if len(collected) >= target:
                    return collected

        if len(batch) < page_size or len(collected) >= total:
            break  # source exhausted

    return collected


def fetch_top_headlines(source_id, page_size=10):
    articles, _ = _get("top-headlines", {"sources": source_id, "pageSize": page_size})
    return articles

## Text Cleaning & Full-Article Extraction

In [7]:
def clean_text(text):
    """Strip HTML, URLs, extra whitespace and NewsAPI truncation markers."""
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\[\+\d+ chars\]", "", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r" {2,}", " ", text).strip()
    return text


def _fetch_html(url):
    """Download raw HTML with a browser-like User-Agent to avoid 403s."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    }
    try:
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()
        return resp.text
    except Exception:
        return None


def _extract_via_trafilatura(html, url):
    """Extract full article text using trafilatura (best recall for news sites)."""
    if not TRAFILATURA_AVAILABLE or not html:
        return {}
    try:
        result = trafilatura.extract(
            html,
            url=url,
            include_comments=False,
            include_tables=False,
            favor_recall=True,       # prioritise getting all body text
            output_format="txt"
        )
        meta = trafilatura.extract_metadata(html, default_url=url)
        return {
            "full_text":    result or "",
            "authors":      list(meta.author.split(";")) if meta and meta.author else [],
            "publish_date": str(meta.date) if meta and meta.date else None
        }
    except Exception:
        return {}


def _extract_via_newspaper(url):
    """Fallback: extract via newspaper3k."""
    if not NEWSPAPER_AVAILABLE:
        return {}
    try:
        art = Article(url, language="en", fetch_images=False)
        art.download()
        art.parse()
        return {
            "full_text":    art.text or "",
            "authors":      art.authors,
            "publish_date": str(art.publish_date) if art.publish_date else None
        }
    except Exception:
        return {}


def extract_full_article(url):
    """
    Return the full article body from a URL.
    Strategy: trafilatura (primary) → newspaper3k (fallback).
    Both get the same downloaded HTML where possible.
    """
    if not url:
        return {}

    html = _fetch_html(url)

    # --- trafilatura pass ---
    result = _extract_via_trafilatura(html, url)
    if result.get("full_text") and len(result["full_text"].split()) >= 50:
        return result

    # --- newspaper3k fallback ---
    result = _extract_via_newspaper(url)
    if result.get("full_text") and len(result["full_text"].split()) >= 50:
        return result

    return result  # return whatever we have even if short

## Fetch All Articles — with Checkpoint & Resume

In [8]:
def save_article(record, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False, default=str)


def load_checkpoint():
    """Load saved progress. Returns dict with 'completed' list of pair keys."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, encoding="utf-8") as f:
            cp = json.load(f)
        n = len(cp.get("completed", []))
        print(f"Checkpoint found — {n} (topic × source) pair(s) already done.")
        return cp
    return {"completed": []}


def save_checkpoint(cp):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(cp, f, indent=2)


def load_existing_articles():
    """
    Read previously saved article JSON files from disk into all_articles.
    Must be called before the fetch loop so skipped pairs still appear in
    the summary and downstream cells. Skips _duplicates_archive/ — those were
    exact re-fetches of articles already represented elsewhere (see the fetch
    loop's exclude_urls note below) and must not count toward "already saved".
    """
    loaded = 0
    for topic in TOPICS:
        slug = topic.lower().replace(" ", "_")
        for source_key in SOURCES:
            folder = os.path.join(OUTPUT_DIR, slug, source_key)
            if not os.path.isdir(folder):
                continue
            for fname in sorted(os.listdir(folder)):
                if fname.endswith(".json"):
                    with open(os.path.join(folder, fname), encoding="utf-8") as f:
                        all_articles[slug][source_key].append(json.load(f))
                    loaded += 1
    return loaded


# ── Initialise ────────────────────────────────────────────────────────────────
all_articles = defaultdict(lambda: defaultdict(list))

existing_count = load_existing_articles()
if existing_count:
    print(f"Loaded {existing_count} existing article(s) from disk.\n")

checkpoint = load_checkpoint()
completed  = set(checkpoint.get("completed", []))

# Global per-source dedup: a URL already saved for ANY topic under a given source
# must never be fetched/saved again under a different topic. Without this, the
# same article can surface under two topic queries for the same outlet (e.g. an
# "energy" article that also matches "global economy") and get saved twice —
# once per topic folder — which is the cross-topic overlap this seeds against.
source_seen_urls = defaultdict(set)
for topic_slug, sources in all_articles.items():
    for source_key, articles in sources.items():
        for a in articles:
            if a.get("url"):
                source_seen_urls[source_key].add(a["url"])

print(f"\nTarget : {TARGET_PER_SOURCE} articles per (topic × source)")
print(f"Pairs already done : {len(completed)} / {len(TOPICS) * len(SOURCES)}\n")

# ── Fetch loop ────────────────────────────────────────────────────────────────
rate_limited = False

for topic in TOPICS:
    if rate_limited:
        print(f"  [skipped '{topic}' — rate limit already hit this session]")
        continue

    topic_slug = topic.lower().replace(" ", "_")
    print(f"\n── Topic: '{topic}' ─────────────────────────")

    for source_key, source_info in SOURCES.items():
        pair_key      = f"{topic_slug}|{source_key}"
        already_saved = len(all_articles[topic_slug][source_key])
        still_needed  = TARGET_PER_SOURCE - already_saved

        if pair_key in completed or still_needed <= 0:
            print(f"  ⏭  {source_info['label']:35s}: skipped ({already_saved} already saved)")
            continue

        # Exclude every URL already saved for this source under ANY topic (not just
        # this pair) — this is what guarantees no article is ever duplicated across
        # topic folders, on top of preventing the same-pair resume bug described
        # in fetch_by_topic's docstring.
        exclude_urls = source_seen_urls[source_key]

        try:
            raw_articles = fetch_by_topic(
                source_info["source_id"], topic,
                target       = still_needed,
                page_size    = PAGE_SIZE,
                max_pages    = MAX_PAGES,
                exclude_urls = exclude_urls,
            )
        except RateLimitError as e:
            print(f"\n⚠️  Rate limit hit: {e}")
            print("Progress saved. Re-run this cell tomorrow to continue from here.")
            save_checkpoint(checkpoint)
            rate_limited = True
            break

        start_idx = already_saved + 1
        new_saved = 0
        for idx, art in enumerate(raw_articles, start=start_idx):
            record = {
                "source":          source_key,
                "source_label":    source_info["label"],
                "country":         source_info["country"],
                "topic":           topic,
                "topic_slug":      topic_slug,
                "title":           art.get("title", ""),
                "description":     clean_text(art.get("description", "")),
                "content_snippet": clean_text(art.get("content", "")),
                "url":             art.get("url", ""),
                "published_at":    art.get("publishedAt", ""),
                "fetched_at":      datetime.utcnow().isoformat(),
                "author":          art.get("author", ""),
                "full_text":       None,
                "authors":         [],
            }

            full = extract_full_article(record["url"])
            if full:
                record["full_text"] = clean_text(full.get("full_text", ""))
                record["authors"]   = full.get("authors", [])
                if not record["published_at"] and full.get("publish_date"):
                    record["published_at"] = full["publish_date"]

            filepath = os.path.join(OUTPUT_DIR, topic_slug, source_key, f"article_{idx:03d}.json")
            save_article(record, filepath)
            all_articles[topic_slug][source_key].append(record)
            if record["url"]:
                source_seen_urls[source_key].add(record["url"])
            new_saved += 1

        total_now = len(all_articles[topic_slug][source_key])
        has_full  = sum(1 for a in all_articles[topic_slug][source_key] if a.get("full_text"))
        print(f"  {source_info['label']:35s}: +{new_saved} new → {total_now} total  ({has_full} with full text)")

        if total_now >= TARGET_PER_SOURCE:
            completed.add(pair_key)
        checkpoint["completed"] = list(completed)
        save_checkpoint(checkpoint)

if not rate_limited:
    print("\n✓ All pairs at or above target. Delete fetch_progress.json to reset.")

Loaded 120 existing article(s) from disk.


Target : 10 articles per (topic × source)
Pairs already done : 0 / 15


── Topic: 'artificial intelligence' ─────────────────────────
  ⏭  BBC News (UK)                      : skipped (10 already saved)
  ⏭  ABC News (Australia)               : skipped (10 already saved)
  ⏭  Associated Press (US)              : skipped (10 already saved)

── Topic: 'public health' ─────────────────────────


  BBC News (UK)                      : +10 new → 10 total  (10 with full text)


  ABC News (Australia)               : +10 new → 10 total  (10 with full text)


  Associated Press (US)              : +10 new → 10 total  (10 with full text)

── Topic: 'elections' ─────────────────────────
  ⏭  BBC News (UK)                      : skipped (10 already saved)
  ⏭  ABC News (Australia)               : skipped (10 already saved)
  ⏭  Associated Press (US)              : skipped (10 already saved)

── Topic: 'global economy' ─────────────────────────
  ⏭  BBC News (UK)                      : skipped (10 already saved)
  ⏭  ABC News (Australia)               : skipped (10 already saved)
  ⏭  Associated Press (US)              : skipped (10 already saved)

── Topic: 'energy' ─────────────────────────
  ⏭  BBC News (UK)                      : skipped (10 already saved)
  ⏭  ABC News (Australia)               : skipped (10 already saved)
  ⏭  Associated Press (US)              : skipped (10 already saved)

✓ All pairs at or above target. Delete fetch_progress.json to reset.


## Top Headlines Snapshot (Optional — one request per source)

In [9]:
print("Fetching top headlines...")
for source_key, source_info in SOURCES.items():
    headlines = fetch_top_headlines(source_info["source_id"], page_size=10)
    path = os.path.join(OUTPUT_DIR, "top_headlines", f"{source_key}.json")
    save_article({"source": source_key, "articles": headlines}, path)
    print(f"  {source_info['label']:30s}: {len(headlines)} headlines saved")
    time.sleep(0.3)

Fetching top headlines...


  BBC News (UK)                 : 10 headlines saved


  ABC News (Australia)          : 10 headlines saved


  Associated Press (US)         : 10 headlines saved


## Coverage Summary

In [10]:
rows = []
for topic_slug, sources in all_articles.items():
    for source_key, articles in sources.items():
        rows.append({
            "Topic":         topic_slug.replace("_", " ").title(),
            "Source":        SOURCES[source_key]["label"],
            "Country":       SOURCES[source_key]["country"],
            "Articles":      len(articles),
            "Full Text (%)": int(100 * sum(1 for a in articles if a.get("full_text")) / max(len(articles), 1))
        })

df_summary = pd.DataFrame(rows)
pivot = df_summary.pivot_table(
    index="Topic", columns="Country", values="Articles", aggfunc="sum"
).fillna(0).astype(int)

print("Articles per topic per country:")
print(pivot.to_string())
print(f"\nTotal articles collected: {df_summary['Articles'].sum()}")

Articles per topic per country:
Country                  AU  GB  US
Topic                              
Artificial Intelligence  10  10  10
Elections                10  10  10
Energy                   10  10  10
Global Economy           10  10  10
Public Health            10  10  10

Total articles collected: 150


## Overlapping Topic Identification

A topic is considered **overlapping** when all three outlets have at least one article for it.

In [11]:
overlapping_topics = {
    slug: sources
    for slug, sources in all_articles.items()
    if all(len(sources.get(sk, [])) > 0 for sk in SOURCES)
}

print(f"Topics with coverage from ALL three outlets: {len(overlapping_topics)} / {len(TOPICS)}")
print()
for slug, sources in overlapping_topics.items():
    counts = ", ".join(f"{sk}={len(sources[sk])}" for sk in SOURCES)
    print(f"  {slug:<25s} → {counts}")

# Save the overlap index
overlap_index = {
    slug: {sk: [a["url"] for a in arts] for sk, arts in sources.items()}
    for slug, sources in overlapping_topics.items()
}
with open(os.path.join(OUTPUT_DIR, "overlap_index.json"), "w") as f:
    json.dump(overlap_index, f, indent=2)
print("\nOverlap index saved to articles/overlap_index.json")

Topics with coverage from ALL three outlets: 5 / 5

  artificial_intelligence   → bbc=10, abc_au=10, ap_us=10
  elections                 → bbc=10, abc_au=10, ap_us=10
  global_economy            → bbc=10, abc_au=10, ap_us=10
  energy                    → bbc=10, abc_au=10, ap_us=10
  public_health             → bbc=10, abc_au=10, ap_us=10

Overlap index saved to articles/overlap_index.json


## Build Flat Dataset CSV

Produces `articles/articles_dataset.csv` — one row per article, ready for NLP pipelines.

In [12]:
records = []
for topic_slug, sources in all_articles.items():
    for source_key, articles in sources.items():
        for art in articles:
            # Prefer full scraped text; fall back to snippet/description
            body = (
                art.get("full_text")
                or art.get("content_snippet")
                or art.get("description")
                or ""
            )
            records.append({
                "topic":        art["topic"],
                "topic_slug":   topic_slug,
                "source_key":   source_key,
                "source":       art["source_label"],
                "country":      art["country"],
                "title":        art["title"],
                "body":         body,
                "word_count":   len(body.split()),
                "url":          art["url"],
                "published_at": art["published_at"],
                "has_full_text": bool(art.get("full_text"))
            })

df = pd.DataFrame(records)

# Drop rows with too little text to be useful for summarization
df = df[df["word_count"] >= 30].reset_index(drop=True)

csv_path = os.path.join(OUTPUT_DIR, "articles_dataset.csv")
df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Dataset saved  : {csv_path}")
print(f"Total rows     : {len(df)}")
print(f"Avg word count : {df['word_count'].mean():.0f}")
print(f"Full-text rows : {df['has_full_text'].sum()} ({100*df['has_full_text'].mean():.0f}%)")
print()
print(df.groupby(["topic", "country"])["title"].count().unstack(fill_value=0).to_string())

Dataset saved  : /Users/david/Documents/MLDS/Sem 4/NLP/FInal Project /articles/articles_dataset.csv
Total rows     : 150
Avg word count : 963
Full-text rows : 150 (100%)

country                  AU  GB  US
topic                              
artificial intelligence  10  10  10
elections                10  10  10
energy                   10  10  10
global economy           10  10  10
public health            10  10  10


## Quick Sanity Check — Sample Article

In [13]:
sample = df[df["has_full_text"]].sample(1).iloc[0] if df["has_full_text"].any() else df.sample(1).iloc[0]

print(f"Topic    : {sample['topic']}")
print(f"Source   : {sample['source']}")
print(f"Title    : {sample['title']}")
print(f"Words    : {sample['word_count']}")
print(f"URL      : {sample['url']}")
print()
print("Body (first 500 chars):")
print(sample["body"][:500])

Topic    : global economy
Source   : ABC News (Australia)
Title    : AI giants on the nose: Why those stellar super earnings are under threat
Words    : 1206
URL      : https://www.abc.net.au/news/2026-06-30/ai-boom-big-tech-investment-drain-market-volatility/106857426

Body (first 500 chars):
analysis Are the wheels falling off the AI investment boom? Tue 30 Jun 2026 at 4:59am , updatedTue 30 Jun 2026 at 6:02am What, if anything, has the past financial year taught us? Not much, apparently, as it draws to a close this afternoon in much the same way it began. Global geopolitics is in the kind of disarray unseen in almost a century, there is social and political upheaval on almost every continent and the global economy is facing serious issues on all fronts. Inflation is up, growth is s


## Output Structure

```
articles/
├── articles_dataset.csv        ← flat NLP-ready dataset
├── overlap_index.json          ← topics covered by all 3 outlets
├── top_headlines/
│   ├── bbc.json
│   ├── abc_au.json
│   └── ap_us.json
├── public_health/
│   ├── bbc/    article_001.json  …
│   ├── abc_au/ article_001.json  …
│   └── ap_us/  article_001.json  …
├── artificial_intelligence/ …
└── … (one folder per topic)
```